# 2.4 圖片資料夾載入

這份 Notebook 建立一個小型圖片資料夾，示範 `image_dataset_from_directory` 的標準用法。

## 1. 載入套件

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

tf.keras.utils.set_random_seed(42)

## 2. 建立示範圖片資料夾

In [ ]:
DATASET_DIR = Path('demo_images')
classes = ['vertical', 'horizontal']
image_size = (32, 32)
rng = np.random.default_rng(42)

for class_name in classes:
    (DATASET_DIR / class_name).mkdir(parents=True, exist_ok=True)

for i in range(40):
    vertical = np.zeros((32, 32, 3), dtype=np.uint8)
    vertical[:, 13:19, 0] = 220
    vertical += rng.integers(0, 25, size=vertical.shape, dtype=np.uint8)
    tf.io.write_file(str(DATASET_DIR / 'vertical' / f'vertical_{i:03d}.png'), tf.io.encode_png(vertical))

    horizontal = np.zeros((32, 32, 3), dtype=np.uint8)
    horizontal[13:19, :, 1] = 220
    horizontal += rng.integers(0, 25, size=horizontal.shape, dtype=np.uint8)
    tf.io.write_file(str(DATASET_DIR / 'horizontal' / f'horizontal_{i:03d}.png'), tf.io.encode_png(horizontal))

print('dataset dir:', DATASET_DIR.resolve())

## 3. 從資料夾建立 train / validation dataset

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.25,
    subset='training',
    seed=42,
    image_size=image_size,
    batch_size=16
)
valid_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.25,
    subset='validation',
    seed=42,
    image_size=image_size,
    batch_size=16
)

print('class names:', train_ds.class_names)

## 4. 檢查 batch 形狀

In [ ]:
for images, labels in train_ds.take(1):
    print('images:', images.shape, images.dtype)
    print('labels:', labels.shape, labels.numpy()[:8])

## 5. 建立簡單 CNN 驗證資料管線

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(8, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=valid_ds, epochs=5, verbose=0)

pd.DataFrame([
    model.evaluate(train_ds, verbose=0, return_dict=True),
    model.evaluate(valid_ds, verbose=0, return_dict=True),
], index=['train', 'valid'])

## 6. 小結

只要資料夾結構清楚，`image_dataset_from_directory` 就能快速建立影像分類資料集。